# Test Kev on this PC

Loads a released Kev checkpoint in-process (no server) and runs a few decision requests.

**Kernel:** select **Python (kev)**, which is the `kev/.venv` environment created with `uv sync`.

Kev's sizes: `jaredpalmer/kev-0.8b` (smallest), `jaredpalmer/kev-4b` (recommended), `jaredpalmer/kev-9b`.
The first load downloads the LoRA adapter plus the Qwen3.5 base model from Hugging Face.

In [1]:
import time, json
import torch
from kev.api import SystemOneRequest, to_record, to_answers
from kev.checkpoint import Checkpoint, LoadOptions
from kev.model import admit

MODEL = "jaredpalmer/kev-0.8b"   # or "jaredpalmer/kev-0.8b" / "jaredpalmer/kev-9b"
DEVICE = "cuda"

## 1. Check the GPU

Kev-4B in bf16 needs about 9 GB for weights plus a few GB for activations (~14 GB resident when serving).
Kev-0.8B needs about 2-3 GB. If free memory is low, stop other GPU processes (e.g. a running `llama-server`) or switch `MODEL` to the 0.8B.

In [2]:
print(torch.__version__, "CUDA", torch.version.cuda, "| available:", torch.cuda.is_available())
free, total = torch.cuda.mem_get_info()
print(f"{torch.cuda.get_device_name()}: {free/2**30:.1f} GiB free of {total/2**30:.1f} GiB")
need = {"kev-0.8b": 3, "kev-4b": 14, "kev-9b": 24}[MODEL.split("/")[-1]]
if free / 2**30 < need:
    print(f"WARNING: {MODEL} needs roughly {need} GiB free. Free GPU memory or pick a smaller MODEL.")

2.8.0+cu128 CUDA 12.8 | available: True
NVIDIA GeForce RTX 5090: 26.1 GiB free of 31.4 GiB


## 2. Load the model (bf16, LoRA merged into the base)

In [3]:
MODEL

'jaredpalmer/kev-0.8b'

In [3]:
t = time.time()
tok, model = Checkpoint(MODEL).load(DEVICE, LoadOptions(dtype=torch.bfloat16))
model.eval()
print(f"loaded {MODEL} in {time.time()-t:.0f}s | temperature T = {model.head.temperature:.2f}")
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/2**30:.1f} GiB")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

In file included from /home/dw/.local/share/uv/python/cpython-3.13.12-linux-x86_64-gnu/include/python3.13/Python.h:14,
                 from /tmp/tmpilgykeqg/cuda_utils.c:5:
/home/dw/.local/share/uv/python/cpython-3.13.12-linux-x86_64-gnu/include/python3.13/pyconfig.h:1967:9: warning: ‘_POSIX_C_SOURCE’ redefined
 1967 | #define _POSIX_C_SOURCE 200809L
      |         ^~~~~~~~~~~~~~~
In file included from /usr/include/bits/libc-header-start.h:33,
                 from /usr/include/stdlib.h:26,
                 from /home/dw/github/decision-models-finetuning/kev/.venv/lib/python3.13/site-packages/triton/backends/nvidia/include/cuda.h:56,
                 from /tmp/tmpilgykeqg/cuda_utils.c:1:
/usr/include/features.h:319:10: note: this is the location of the previous definition
  319 | # define _POSIX_C_SOURCE        202405L
      |          ^~~~~~~~~~~~~~~


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded jaredpalmer/kev-0.8b in 5s | temperature T = 2.35
GPU memory allocated: 1.4 GiB


In [7]:
# model

## 3. A `decide()` helper

Takes the same request shape as the Jev / Kev `/v1/systemone` API and returns the same `answers` object.

In [8]:
@torch.no_grad()
def decide(state, questions):
    req = SystemOneRequest.model_validate({"state": state, "questions": questions})
    rec, meta = to_record(req)
    enc = admit(model, tok, rec)
    torch.cuda.synchronize(); t = time.perf_counter()
    ps = model.probs(enc)
    torch.cuda.synchronize(); ms = 1000 * (time.perf_counter() - t)
    answers = to_answers([torch.as_tensor(p).float().tolist() for p in ps], meta)
    return {"answers": answers, "input_tokens": len(enc["ids"]), "latency_ms": round(ms, 1)}

def show(out):
    print(json.dumps(out, indent=2))

## 5. Persona -> opinion (our use case, before any fine-tuning)

The state is a demographic persona; the question is a survey item (GSS-style). This shows what the **un-tuned** model
does: it was trained on classification and policy tasks, not on predicting human opinions, so treat these numbers
as a baseline, not as a measurement of public opinion.

In [13]:
personas = {
    "young urban liberal": "A 24-year-old woman living in Seattle, Washington. College graduate, works in tech, not religious, identifies as a liberal Democrat.",
    "older rural conservative": "A 67-year-old man living in rural Alabama. High-school education, retired farmer, attends a Baptist church weekly, identifies as a conservative Republican.",
    "middle-aged moderate": "A 45-year-old man living in suburban Ohio. Some college, works as an electrician, Catholic, identifies as a political Independent.",
}
item = {
    "environment": {"type": "score",
                    "instructions": "How would this person answer: 'Is the nation spending too much, too little, or about the right amount on improving and protecting the environment?'",
                    "criteria": ["Too little", "About right", "Too much"]},
    "gun_permit": {"type": "noul",
                   "instructions": "Would this person favor a law banning gun ownership?"},
}
for name, p in personas.items():
    a = decide(p, item)["answers"]
    env = a["environment"]
    print(f"{name:26s} environment score={env['score']:.2f} probs={env['probabilities']} | P(favor permit)={a['gun_permit']['noul']:.2f}")

young urban liberal        environment score=0.91 probs={'0': 0.279, '1': 0.5311, '2': 0.1898} | P(favor permit)=0.46
older rural conservative   environment score=0.89 probs={'0': 0.3193, '1': 0.475, '2': 0.2058} | P(favor permit)=0.46
middle-aged moderate       environment score=0.93 probs={'0': 0.2938, '1': 0.4861, '2': 0.2201} | P(favor permit)=0.49


In [14]:
env

{'type': 'score',
 'score': 0.9263,
 'legend': {'0': 'Too little', '1': 'About right', '2': 'Too much'},
 'probabilities': {'0': 0.2938, '1': 0.4861, '2': 0.2201},
 'confidence': 0.2291}

In [16]:
a

{'environment': {'type': 'score',
  'score': 0.9263,
  'legend': {'0': 'Too little', '1': 'About right', '2': 'Too much'},
  'probabilities': {'0': 0.2938, '1': 0.4861, '2': 0.2201},
  'confidence': 0.2291},
 'gun_permit': {'type': 'noul', 'noul': 0.492}}

## 6. Sanity check: option-order sensitivity

Kev's README says option order can change answers. Same persona and question, options reversed.

In [17]:
p = personas["middle-aged moderate"]
q = "Which statement comes closer to this person's view?"
fwd = {"gov": {"type": "choice", "instructions": q, "criteria": {
    "more": "Government should do more to solve problems", "less": "Government is doing too many things better left to businesses and individuals"}}}
rev = {"gov": {"type": "choice", "instructions": q, "criteria": dict(reversed(list(fwd["gov"]["criteria"].items())))}}
print("original order:", decide(p, fwd)["answers"]["gov"]["probabilities"])
print("reversed order:", decide(p, rev)["answers"]["gov"]["probabilities"])

original order: {'more': 0.3763, 'less': 0.6237}
reversed order: {'less': 0.4118, 'more': 0.5882}


In [18]:
fwd

{'gov': {'type': 'choice',
  'instructions': "Which statement comes closer to this person's view?",
  'criteria': {'more': 'Government should do more to solve problems',
   'less': 'Government is doing too many things better left to businesses and individuals'}}}

In [19]:
rev

{'gov': {'type': 'choice',
  'instructions': "Which statement comes closer to this person's view?",
  'criteria': {'less': 'Government is doing too many things better left to businesses and individuals',
   'more': 'Government should do more to solve problems'}}}

## 7. Speed

In [20]:
for _ in range(2): decide(personas["young urban liberal"], item)   # warm-up
lat = [decide(personas["young urban liberal"], item)["latency_ms"] for _ in range(10)]
print(f"median latency over 10 requests: {sorted(lat)[5]:.0f} ms")
print(f"peak GPU memory: {torch.cuda.max_memory_allocated()/2**30:.1f} GiB")

median latency over 10 requests: 32 ms
peak GPU memory: 1.8 GiB


## Next

Fine-tuning on our own labels uses the same request shape plus a `label` per question (see `kev/README.md`,
"Fine-Tune on Your Own Data"):

```bash
uv run python -m kev.train --data train.jsonl --base Qwen/Qwen3.5-4B-Base --init_from jaredpalmer/kev-4b ...
```